In [0]:
# Code Generated by Sidekick is for learning and experimentation purposes only.

import requests
import json
import logging
from datetime import datetime
from typing import Generator

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("fhir_utils")


def build_fhir_url(base_url: str, resource: str, start_date: str, end_date: str,
                   fmt: str = "json") -> str:
    """Build initial FHIR search URL with date filter and format."""
    date_param = (
        f"_lastUpdated=ge{start_date}&_lastUpdated=le{end_date}"
        if resource != "Patient"
        else ""
    )
    return (
        f"{base_url}/{resource}"
        f"?_format={fmt}&_count=50"
        + (f"&{date_param}" if date_param else "")
    )


def paginate_fhir(url: str, headers: dict = None) -> Generator[dict, None, None]:
    """
    Generator: yields one bundle dict per page.
    Follows 'next' links automatically.
    """
    session   = requests.Session()
    next_url  = url
    page_num  = 0

    while next_url:
        page_num += 1
        logger.info(f"Fetching page {page_num}: {next_url}")
        resp = session.get(next_url, headers=headers or {}, timeout=30)
        resp.raise_for_status()

        bundle    = resp.json()
        yield bundle

        # Follow FHIR 'next' link
        next_url  = next(
            (link["url"] for link in bundle.get("link", []) if link["relation"] == "next"),
            None,
        )


def extract_entries(bundle: dict) -> list:
    """Return list of resource dicts from a FHIR bundle."""
    return [entry["resource"] for entry in bundle.get("entry", []) if "resource" in entry]


def add_metadata(records: list, api_url: str, resource_type: str) -> list:
    """Attach extraction metadata to every record."""
    ts = datetime.utcnow().isoformat()
    for rec in records:
        rec["_extraction_timestamp"] = ts
        rec["_api_url"]              = api_url
        rec["_resource_type"]        = resource_type
        rec["_ingestion_date"]       = ts[:10]   # YYYY-MM-DD partition key
    return records
